# Deep Space, Deeper Learning
## Galaxy Morphology Classifier with Convolutional Neural Networks
*CPSC 381/581: Machine Learning — Yale University — Instructor: Alex Wong*

**Authors:** Jeet Parikh, Abheek Dhawan

---

This notebook trains a CNN to classify galaxy images from the Sloan Digital Sky Survey (SDSS)
into three morphological types using citizen-labeled data from Galaxy Zoo 2:

| Label | Class | Description |
|-------|-------|-------------|
| 0 | **Smooth** | Elliptical / lenticular — featureless, round |
| 1 | **Disk** | Spiral — clearly shows disk structure or arms |
| 2 | **Irregular** | Mergers, peculiar, or ambiguous morphology |

We train two models:
1. A custom baseline CNN (built from scratch)
2. A fine-tuned ResNet-18 using transfer learning


## 0. Setup

In [ ]:
# Mount Drive and cd into repo
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
os.chdir('/content/drive/MyDrive/cpsc3810-galaxy-classifier')
print("Working directory:", os.getcwd())


In [ ]:
!git pull   # pull any updates from the repo
!pip install -r requirements.txt -q


In [ ]:
# Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found. Training will be slow. Change runtime to GPU.")


## 1. Hyperparameters

All training hyperparameters are defined here so they're easy to find and report.


In [ ]:
# ── Model selection ────────────────────────────────────────────────────────────
MODEL_NAME = 'resnet'      # 'baseline' or 'resnet'

# ── Data ───────────────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
NUM_WORKERS  = 2           # keep at 2 for Colab
VAL_FRAC     = 0.15        # 15% validation
TEST_FRAC    = 0.15        # 15% test
SEED         = 42

# ── Optimiser ──────────────────────────────────────────────────────────────────
HEAD_LR      = 1e-3        # learning rate for classification head
BACKBONE_LR  = 1e-4        # learning rate for backbone (Phase B only)
WEIGHT_DECAY = 1e-4        # L2 regularisation
MOMENTUM     = 0.9         # for SGD

# ── Training schedule ──────────────────────────────────────────────────────────
EPOCHS_A     = 10          # Phase A: head-only (frozen backbone)
EPOCHS_B     = 30          # Phase B: full fine-tune (ResNet only)
PATIENCE     = 7           # early stopping patience (val loss)
LR_DECAY     = 0.5         # ReduceLROnPlateau reduction factor
LR_PATIENCE  = 3           # epochs before reducing LR

print("Hyperparameters set.")


## 2. Imports

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Imports complete.")


## 3. Data

### 3.1 Paths
Point these at wherever you unzipped the galaxy data.


In [ ]:
DATA_DIR   = Path('data')
IMAGE_DIR  = DATA_DIR / 'images'
LABEL_FILE = DATA_DIR / 'labels.csv'
OUT_DIR    = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

assert LABEL_FILE.exists(), f"labels.csv not found at {LABEL_FILE}. Run data/download.py first."
assert IMAGE_DIR.exists(),  f"images/ not found at {IMAGE_DIR}."

df_all = pd.read_csv(LABEL_FILE)
df_all = df_all[df_all['objid'].apply(lambda x: (IMAGE_DIR / f'{x}.jpg').exists())]
print(f"Total images found on disk: {len(df_all):,}")
print(df_all['label'].value_counts().rename({0: 'Smooth', 1: 'Disk', 2: 'Irregular'}))


### 3.2 Class metadata

In [ ]:
CLASS_NAMES  = ['Smooth', 'Disk', 'Irregular']
NUM_CLASSES  = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


### 3.3 Visualise sample images

Let's look at a few examples from each class before training.


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 7))
fig.suptitle('Sample Galaxy Images by Class', fontsize=14, fontweight='bold')

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_df = df_all[df_all['label'] == class_idx].sample(6, random_state=SEED)
    for col, (_, row) in enumerate(class_df.iterrows()):
        img = Image.open(IMAGE_DIR / f"{row['objid']}.jpg").convert('RGB')
        axes[class_idx, col].imshow(img)
        axes[class_idx, col].axis('off')
        if col == 0:
            axes[class_idx, col].set_ylabel(class_name, fontsize=12, fontweight='bold', rotation=90, labelpad=40)

plt.tight_layout()
plt.savefig(OUT_DIR / 'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()


### 3.4 Transforms

In [ ]:
def get_transforms(split):
    '''
    Return the image transform pipeline for each dataset split.

    Training augmentation rationale:
      - Random horizontal/vertical flip + 360-degree rotation:
        galaxies have no canonical orientation, all rotations are valid.
      - ColorJitter: handles variation in SDSS photometric calibration.

    Arg(s):
        split : str
            One of 'train', 'val', 'test'
    Returns:
        torchvision.transforms.Compose
    '''
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(180),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

print("Transforms defined.")


### 3.5 Dataset class

In [ ]:
class GalaxyDataset(Dataset):
    '''
    PyTorch Dataset for galaxy morphology images.

    Arg(s):
        df        : pd.DataFrame  — columns ['objid', 'label']
        image_dir : Path          — directory containing <objid>.jpg files
        split     : str           — 'train', 'val', or 'test'
    '''

    def __init__(self, df, image_dir, split='train'):
        self.df        = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = get_transforms(split)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = self.image_dir / f"{row['objid']}.jpg"
        image = Image.open(path).convert('RGB')
        image = self.transform(image)
        label = int(row['label'])
        return image, label

print("GalaxyDataset defined.")


### 3.6 Train / val / test split and DataLoaders

In [ ]:
# Stratified split — done before any augmentation
train_df, temp_df = train_test_split(
    df_all, test_size=(VAL_FRAC + TEST_FRAC),
    stratify=df_all['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
    stratify=temp_df['label'], random_state=SEED
)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

# Weighted sampler — up-samples rare classes for balanced mini-batches
class_counts  = train_df['label'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_df['label'].values]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_df),
    replacement=True,
)

train_ds = GalaxyDataset(train_df, IMAGE_DIR, split='train')
val_ds   = GalaxyDataset(val_df,   IMAGE_DIR, split='val')
test_ds  = GalaxyDataset(test_df,  IMAGE_DIR, split='test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoaders ready.")


## 4. Model Definitions

### 4.1 Baseline CNN

A custom 4-layer convolutional network built from scratch.
We train this first to establish a baseline before using transfer learning.


In [ ]:
class ConvBlock(nn.Module):
    '''Convolution -> BatchNorm -> ReLU -> MaxPool block.'''

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class BaselineCNN(nn.Module):
    '''
    4-layer CNN for 128x128 RGB galaxy images.

    Architecture:
        ConvBlock(3->32) -> ConvBlock(32->64) ->
        ConvBlock(64->128) -> ConvBlock(128->256) ->
        AdaptiveAvgPool -> FC(256->128) -> Dropout -> FC(128->n_class)

    Arg(s):
        num_classes : int  — number of output classes
        dropout     : float — dropout probability before output layer
    '''

    def __init__(self, num_classes=NUM_CLASSES, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

print("BaselineCNN defined.")


### 4.2 ResNet-18 with Transfer Learning

We load ImageNet pretrained weights and replace the final classification head.

**Two-phase training strategy:**
- **Phase A** (frozen backbone): only the new FC head is trained, preventing
  large early gradients from corrupting pretrained features.
- **Phase B** (unfrozen backbone): full end-to-end fine-tuning at a lower
  backbone learning rate.


In [ ]:
def build_resnet(num_classes=NUM_CLASSES, freeze_backbone=True, pretrained=True):
    '''
    Load pretrained ResNet-18 and replace the final FC layer.

    Arg(s):
        num_classes     : int  — number of output classes
        freeze_backbone : bool — if True, freeze all layers except FC head
        pretrained      : bool — use ImageNet pretrained weights
    Returns:
        model : nn.Module
    '''
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.resnet18(weights=weights)

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes),
    )

    if freeze_backbone:
        for name, param in model.named_parameters():
            if not name.startswith('fc'):
                param.requires_grad = False

    return model


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total:,}  |  Trainable: {trainable:,}")

print("ResNet builder defined.")


## 5. Training Loop

In [ ]:
def train_one_epoch(model, dataloader, optimizer, loss_func, device):
    '''
    Run one full pass over the training set.

    Arg(s):
        model      : nn.Module
        dataloader : DataLoader
        optimizer  : torch.optim
        loss_func  : nn.Module      — cross-entropy loss
        device     : torch.device
    Returns:
        mean_loss : float
        accuracy  : float
    '''
    model.train()
    total_loss, n_correct, n_sample = 0.0, 0, 0

    for images, labels in tqdm(dataloader, desc='  train', leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = loss_func(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        n_correct  += (outputs.argmax(1) == labels).sum().item()
        n_sample   += len(labels)

    return total_loss / n_sample, n_correct / n_sample


def evaluate(model, dataloader, loss_func, class_names, device):
    '''
    Evaluate the model on a dataset split.

    Arg(s):
        model      : nn.Module
        dataloader : DataLoader
        loss_func  : nn.Module
        class_names: list[str]
        device     : torch.device
    Returns:
        mean_loss : float
        accuracy  : float
    '''
    model.eval()
    total_loss, n_correct, n_sample = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='  eval ', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss    = loss_func(outputs, labels)

            total_loss += loss.item() * len(labels)
            n_correct  += (outputs.argmax(1) == labels).sum().item()
            n_sample   += len(labels)

    mean_accuracy = 100.0 * n_correct / n_sample
    print(f'Mean accuracy over {n_sample} images: {mean_accuracy:.3f}%')
    return total_loss / n_sample, n_correct / n_sample


print("Training functions defined.")


In [ ]:
def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              loss_func, n_epoch, patience, checkpoint_path, phase_name,
              history, device):
    '''
    Training phase loop with early stopping and best-checkpoint saving.

    Arg(s):
        model           : nn.Module
        train_loader    : DataLoader
        val_loader      : DataLoader
        optimizer       : torch.optim
        scheduler       : LR scheduler
        loss_func       : nn.Module
        n_epoch         : int     — max epochs for this phase
        patience        : int     — early stopping patience
        checkpoint_path : Path    — where to save best weights
        phase_name      : str     — 'A' or 'B' (for logging)
        history         : list    — appended in place
        device          : torch.device
    '''
    best_val_loss = float('inf')
    wait = 0

    for epoch in range(1, n_epoch + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_func, device)
        val_loss,   val_acc   = evaluate(model, val_loader, loss_func, CLASS_NAMES, device)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        history.append({
            'phase': phase_name, 'epoch': epoch,
            'train_loss': round(train_loss, 4), 'train_acc': round(train_acc, 4),
            'val_loss':   round(val_loss,   4), 'val_acc':   round(val_acc,   4),
        })

        print(
            f'[Phase {phase_name}] Epoch {epoch:3d}/{n_epoch} | '
            f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f} | '
            f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f} | '
            f'{elapsed:.1f}s'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), checkpoint_path)
            wait = 0
            print(f'             ↳ Best model saved (val_loss={best_val_loss:.4f})')
        else:
            wait += 1
            if wait >= patience:
                print(f'[Phase {phase_name}] Early stopping at epoch {epoch}')
                break

print("run_phase defined.")


## 6. Train Baseline CNN

We train our custom CNN first to establish a baseline accuracy.


In [ ]:
baseline_ckpt   = OUT_DIR / 'best_baseline.pt'
baseline_history = []

baseline_model = BaselineCNN(num_classes=NUM_CLASSES).to(device)
count_params(baseline_model)

loss_func = nn.CrossEntropyLoss()

optimizer_base = torch.optim.SGD(
    baseline_model.parameters(),
    lr=HEAD_LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
scheduler_base = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_base, patience=LR_PATIENCE, factor=LR_DECAY, verbose=True
)

run_phase(
    baseline_model, train_loader, val_loader,
    optimizer_base, scheduler_base, loss_func,
    n_epoch=EPOCHS_A + EPOCHS_B,
    patience=PATIENCE,
    checkpoint_path=baseline_ckpt,
    phase_name='Baseline',
    history=baseline_history,
    device=device,
)

with open(OUT_DIR / 'history_baseline.json', 'w') as f:
    json.dump(baseline_history, f, indent=2)
print("Baseline training complete.")


## 7. Train ResNet-18 (Transfer Learning)

### Phase A — Head only (frozen backbone)


In [ ]:
resnet_ckpt   = OUT_DIR / 'best_resnet.pt'
resnet_history = []

resnet_model = build_resnet(num_classes=NUM_CLASSES, freeze_backbone=True, pretrained=True)
resnet_model = resnet_model.to(device)
count_params(resnet_model)

optimizer_a = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY,
)
scheduler_a = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_a, patience=LR_PATIENCE, factor=LR_DECAY, verbose=True
)

run_phase(
    resnet_model, train_loader, val_loader,
    optimizer_a, scheduler_a, loss_func,
    n_epoch=EPOCHS_A, patience=PATIENCE,
    checkpoint_path=resnet_ckpt,
    phase_name='A', history=resnet_history, device=device,
)


### Phase B — Full fine-tuning (unfrozen backbone)

In [ ]:
# Load best Phase A weights before unfreezing
resnet_model.load_state_dict(torch.load(resnet_ckpt, map_location=device))

# Unfreeze all parameters
for param in resnet_model.parameters():
    param.requires_grad = True
count_params(resnet_model)

# Differential learning rates: lower LR for backbone, higher for head
backbone_params = [p for n, p in resnet_model.named_parameters() if not n.startswith('fc')]
head_params     = [p for n, p in resnet_model.named_parameters() if     n.startswith('fc')]

optimizer_b = torch.optim.Adam([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params,     'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b, patience=LR_PATIENCE, factor=LR_DECAY, verbose=True
)

run_phase(
    resnet_model, train_loader, val_loader,
    optimizer_b, scheduler_b, loss_func,
    n_epoch=EPOCHS_B, patience=PATIENCE,
    checkpoint_path=resnet_ckpt,
    phase_name='B', history=resnet_history, device=device,
)

with open(OUT_DIR / 'history_resnet.json', 'w') as f:
    json.dump(resnet_history, f, indent=2)
print("ResNet training complete.")


## 8. Evaluation

### 8.1 Helper — collect predictions


In [ ]:
def get_predictions(model, dataloader, device):
    '''
    Run inference and collect predictions and ground-truth labels.

    Arg(s):
        model      : nn.Module
        dataloader : DataLoader
        device     : torch.device
    Returns:
        y_pred  : np.ndarray of shape (N,)
        y_true  : np.ndarray of shape (N,)
    '''
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Inference', leave=False):
            images  = images.to(device)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)

print("get_predictions defined.")


### 8.2 Evaluate both models on test set

In [ ]:
# Load best checkpoints
baseline_model.load_state_dict(torch.load(baseline_ckpt, map_location=device))
resnet_model.load_state_dict(torch.load(resnet_ckpt, map_location=device))

print("=== Baseline CNN ===")
baseline_preds, baseline_true = get_predictions(baseline_model, test_loader, device)
baseline_acc = accuracy_score(baseline_true, baseline_preds)
baseline_f1  = f1_score(baseline_true, baseline_preds, average='weighted')
print(f"Mean accuracy over {len(baseline_true)} images: {baseline_acc*100:.3f}%")
print(f"Weighted F1: {baseline_f1:.4f}")
print()
print(classification_report(baseline_true, baseline_preds, target_names=CLASS_NAMES))

print("\n=== ResNet-18 ===")
resnet_preds, resnet_true = get_predictions(resnet_model, test_loader, device)
resnet_acc = accuracy_score(resnet_true, resnet_preds)
resnet_f1  = f1_score(resnet_true, resnet_preds, average='weighted')
print(f"Mean accuracy over {len(resnet_true)} images: {resnet_acc*100:.3f}%")
print(f"Weighted F1: {resnet_f1:.4f}")
print()
print(classification_report(resnet_true, resnet_preds, target_names=CLASS_NAMES))

random_baseline = 1.0 / NUM_CLASSES
print(f"Random baseline: {random_baseline*100:.1f}%")


### 8.3 Confusion matrices

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, ax_raw, ax_norm):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax_raw, linewidths=0.5)
    ax_raw.set_xlabel('Predicted'); ax_raw.set_ylabel('True')
    ax_raw.set_title(f'{title} — Raw counts')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax_norm, linewidths=0.5)
    ax_norm.set_xlabel('Predicted'); ax_norm.set_ylabel('True')
    ax_norm.set_title(f'{title} — Normalised')


fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

plot_confusion_matrix(baseline_true, baseline_preds, 'Baseline CNN',
                      axes[0][0], axes[0][1])
plot_confusion_matrix(resnet_true,   resnet_preds,   'ResNet-18',
                      axes[1][0], axes[1][1])

plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.4 Training curves

In [ ]:
def plot_history(history, title, ax_loss, ax_acc):
    epochs     = [r['epoch']      for r in history]
    train_loss = [r['train_loss'] for r in history]
    val_loss   = [r['val_loss']   for r in history]
    train_acc  = [r['train_acc']  for r in history]
    val_acc    = [r['val_acc']    for r in history]

    ax_loss.plot(epochs, train_loss, label='Train', marker='o', markersize=3)
    ax_loss.plot(epochs, val_loss,   label='Val',   marker='o', markersize=3)
    ax_loss.set_title(f'{title} — Loss'); ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Cross-entropy loss'); ax_loss.legend()

    ax_acc.plot(epochs, train_acc, label='Train', marker='o', markersize=3)
    ax_acc.plot(epochs, val_acc,   label='Val',   marker='o', markersize=3)
    ax_acc.set_title(f'{title} — Accuracy'); ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy'); ax_acc.set_ylim(0, 1); ax_acc.legend()

    # Phase boundary for ResNet
    phases = [r.get('phase', '') for r in history]
    for i in range(1, len(phases)):
        if phases[i] != phases[i-1]:
            ax_loss.axvline(x=epochs[i], color='gray', linestyle='--', alpha=0.5)
            ax_acc.axvline( x=epochs[i], color='gray', linestyle='--', alpha=0.5,
                            label='Phase B start')


fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

plot_history(baseline_history, 'Baseline CNN', axes[0][0], axes[0][1])
plot_history(resnet_history,   'ResNet-18',    axes[1][0], axes[1][1])

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.5 Per-class F1 comparison

In [ ]:
baseline_report = classification_report(baseline_true, baseline_preds,
                                        target_names=CLASS_NAMES, output_dict=True)
resnet_report   = classification_report(resnet_true,   resnet_preds,
                                        target_names=CLASS_NAMES, output_dict=True)

x = np.arange(NUM_CLASSES)
width = 0.35
colors = ['#4878CF', '#6ACC65']

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, [baseline_report[c]['f1-score'] for c in CLASS_NAMES],
               width, label='Baseline CNN', color=colors[0], alpha=0.85)
bars2 = ax.bar(x + width/2, [resnet_report[c]['f1-score']   for c in CLASS_NAMES],
               width, label='ResNet-18',   color=colors[1], alpha=0.85)

for bars in (bars1, bars2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.1); ax.set_ylabel('F1 Score')
ax.set_title('Per-class F1 Score — Baseline vs ResNet-18', fontweight='bold')
ax.axhline(y=1/NUM_CLASSES, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'per_class_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Grad-CAM Visualisation (Stretch Goal)

Grad-CAM (Selvaraju et al., 2017) shows which spatial regions of the galaxy
image most influenced the model's prediction.


In [ ]:
class GradCAM:
    '''
    Gradient-weighted Class Activation Mapping.

    Hooks into a target convolutional layer and computes a weighted
    combination of its feature maps guided by the gradients flowing
    back from the target class score.

    Arg(s):
        model        : nn.Module
        target_layer : nn.Module — last conv layer to hook into
    '''

    def __init__(self, model, target_layer):
        self.model        = model
        self.gradients    = None
        self.activations  = None
        target_layer.register_forward_hook(
            lambda _, __, out: setattr(self, 'activations', out.detach())
        )
        target_layer.register_full_backward_hook(
            lambda _, __, grad_out: setattr(self, 'gradients', grad_out[0].detach())
        )

    def generate(self, input_tensor, target_class=None):
        '''
        Compute the Grad-CAM heatmap.

        Arg(s):
            input_tensor : torch.Tensor — (1, C, H, W)
            target_class : int or None  — class to explain (default: predicted)
        Returns:
            cam        : np.ndarray (H, W) in [0, 1]
            pred_class : int
            probs      : np.ndarray (num_classes,)
        '''
        self.model.eval()
        inp = input_tensor.to(device).requires_grad_(True)

        logits = self.model(inp)
        probs  = F.softmax(logits, dim=1).squeeze().cpu().detach().numpy()
        pred_class = int(np.argmax(probs))
        if target_class is None:
            target_class = pred_class

        self.model.zero_grad()
        logits[0, target_class].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1).squeeze()
        cam     = F.relu(torch.tensor(cam.numpy())).numpy()
        if cam.max() > 0:
            cam = cam / cam.max()

        h, w = input_tensor.shape[2], input_tensor.shape[3]
        cam = np.array(Image.fromarray(cam).resize((w, h), resample=Image.BILINEAR))
        return cam, pred_class, probs


def denormalize(tensor):
    mean = np.array(IMAGENET_MEAN)[:, None, None]
    std  = np.array(IMAGENET_STD)[:, None, None]
    img  = tensor.cpu().numpy() * std + mean
    return np.clip(img.transpose(1, 2, 0), 0, 1)


def show_gradcam_grid(model, dataset, n_per_class=2):
    '''
    Show a grid of Grad-CAM visualisations, one row per class.

    Arg(s):
        model       : nn.Module (ResNet)
        dataset     : GalaxyDataset
        n_per_class : int — number of examples per class
    '''
    gcam = GradCAM(model, model.layer4[-1])
    val_tf = get_transforms('val')

    fig, axes = plt.subplots(NUM_CLASSES * n_per_class, 3,
                             figsize=(10, 4 * NUM_CLASSES * n_per_class))
    fig.suptitle('Grad-CAM: Regions influencing predictions', fontsize=13, fontweight='bold')

    col_titles = ['Original', 'Grad-CAM heatmap', 'Overlay']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    row = 0
    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_df = dataset.df[dataset.df['label'] == class_idx].sample(
            n_per_class, random_state=SEED)
        for _, record in class_df.iterrows():
            img_pil = Image.open(IMAGE_DIR / f"{record['objid']}.jpg").convert('RGB')
            inp_t   = val_tf(img_pil).unsqueeze(0)
            cam, pred_class, probs = gcam.generate(inp_t)

            img_np  = denormalize(inp_t.squeeze())
            heatmap = plt.cm.jet(cam)[..., :3]
            overlay = np.clip(0.5 * img_np + 0.5 * heatmap, 0, 1)

            correct = (pred_class == class_idx)
            color   = 'green' if correct else 'red'
            label   = (f"True: {class_name}\n"
                       f"Pred: {CLASS_NAMES[pred_class]} ({probs[pred_class]*100:.0f}%)")

            axes[row, 0].imshow(img_np)
            axes[row, 0].set_ylabel(label, fontsize=8, color=color,
                                    rotation=0, labelpad=80, va='center')
            axes[row, 1].imshow(cam, cmap='jet')
            axes[row, 2].imshow(overlay)

            for ax in axes[row]:
                ax.axis('off')
            row += 1

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'gradcam_grid.png', dpi=150, bbox_inches='tight')
    plt.show()


show_gradcam_grid(resnet_model, val_ds, n_per_class=2)


## 10. Results Summary

In [ ]:
print("=" * 55)
print("  Final Results Summary")
print("=" * 55)
print(f"  Random baseline     : {1/NUM_CLASSES*100:.1f}%")
print(f"  Baseline CNN        : {baseline_acc*100:.2f}%  (F1={baseline_f1:.4f})")
print(f"  ResNet-18           : {resnet_acc*100:.2f}%  (F1={resnet_f1:.4f})")
print("=" * 55)

# Save summary JSON
summary = {
    'random_baseline': round(1/NUM_CLASSES, 4),
    'baseline_cnn':  {'accuracy': round(baseline_acc, 4), 'weighted_f1': round(baseline_f1, 4)},
    'resnet18':      {'accuracy': round(resnet_acc,   4), 'weighted_f1': round(resnet_f1,   4)},
}
with open(OUT_DIR / 'results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"\nAll outputs saved to {OUT_DIR}/")


## 11. Export Notebook to PDF (for submission)

In [ ]:
!apt-get -qq install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import files

notebook_path = '/content/drive/MyDrive/cpsc3810-galaxy-classifier/galaxy_classifier.ipynb'
output_pdf    = notebook_path.replace('.ipynb', '.pdf')

!jupyter nbconvert --to pdf "{notebook_path}"
files.download(output_pdf)
